Download each CSV file for each month

In [33]:
import requests

for year in range(2026, 2027):
    for month in range(1,13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        open(f"ontime_{year}_{month}.zip", "wb").write(r.content)

Import some libraries, including snowflake connector

In [ ]:
import io, gzip, zipfile, snowflake.connector

Create the connection to snowflake and put the credetials in a separate file.
Also created a warehouse in snowflake, a database and a schema to store the data.

In [16]:
import credentials
conn = snowflake.connector.connect(
    user=credentials.credentials['user'],
    password=credentials.credentials['password'],
    account=credentials.credentials['account'],
    warehouse=credentials.credentials['warehouse'],
    database=credentials.credentials['database'],
    schema=credentials.credentials['schema']
)

Before the next step I also created a STAGE inside snowflake to store the stage data or raw data.

Unzip the downloaded files locally and push them to the snowflake staging area called @staging_data. 

In [22]:
import os, re, io, gzip, zipfile
for file in os.listdir():
    print(f"Found {file}")
    if file.startswith("ontime_") and file.endswith(".zip"):
        print(f"Processing {file} ...")
        with open(file, "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        # clean output name: no parentheses, ends in .csv 
                        out = re.sub(r"[()]", "", os.path.basename(name)) + ".gz"
                        with z.open(name) as csv_file, gzip.open(out, "wt", encoding="utf-8", newline="") as dst:
                            for line in io.TextIOWrapper(csv_file, encoding="utf-8", errors="replace", newline=""):
                                line = line.rstrip("\r\n")
                                if line.endswith(","):
                                    line = line[:-1]          # drop only the one trailing comma
                                dst.write(line + "\n")
                        print(f"Uploading {out}")
                        conn.cursor().execute(f"PUT 'file://{out}' @staging_data AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

Processing ontime_2025_8.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.csv.gz
Processing ontime_2025_9.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.csv.gz
Processing ontime_2025_4.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_4.csv.gz
Processing ontime_2025_5.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.csv.gz
Processing ontime_2025_7.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.csv.gz
Processing ontime_2025_6.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_6.csv.gz
Processing ontime_2025_2.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_2.csv.gz
Processing ontime_2025_10.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_10.csv.gz
Processing ontime_2025_11.zip ...
Uploading On_Time_Reporting_Carrier_

After we successfully moved the files to snowflake, now we need to remove them locally.

In [ ]:
for file in os.listdir():
    if file.endswith("gz") or file.endswith("zip"):
        print(f"Removing {file} ...")
        os.remove(file)

Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_3.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_12.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_2.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_10.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_6.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_4.csv.gz ...


Created this table in snowflake with data from staging:

CREATE TABLE "AVIATION_WEATHER_PROJECT"."PUBLIC"."FLIGHTS" ( Year NUMBER(38, 0) , Quarter NUMBER(38, 0) , Month NUMBER(38, 0) , DayofMonth NUMBER(38, 0) , DayOfWeek NUMBER(38, 0) , FlightDate DATE , Reporting_Airline VARCHAR , DOT_ID_Reporting_Airline NUMBER(38, 0) , IATA_CODE_Reporting_Airline VARCHAR , Tail_Number VARCHAR , Flight_Number_Reporting_Airline NUMBER(38, 0) , OriginAirportID NUMBER(38, 0) , OriginAirportSeqID NUMBER(38, 0) , OriginCityMarketID NUMBER(38, 0) , Origin VARCHAR , OriginCityName VARCHAR , OriginState VARCHAR , OriginStateFips NUMBER(38, 0) , OriginStateName VARCHAR , OriginWac NUMBER(38, 0) , DestAirportID NUMBER(38, 0) , DestAirportSeqID NUMBER(38, 0) , DestCityMarketID NUMBER(38, 0) , Dest VARCHAR , DestCityName VARCHAR , DestState VARCHAR , DestStateFips NUMBER(38, 0) , DestStateName VARCHAR , DestWac NUMBER(38, 0) , CRSDepTime NUMBER(38, 0) , DepTime VARCHAR , DepDelay NUMBER(38, 2) , DepDelayMinutes NUMBER(38, 2) , DepDel15 NUMBER(38, 2) , DepartureDelayGroups NUMBER(38, 0) , DepTimeBlk VARCHAR , TaxiOut NUMBER(38, 2) , WheelsOff VARCHAR , WheelsOn VARCHAR , TaxiIn NUMBER(38, 2) , CRSArrTime NUMBER(38, 0) , ArrTime VARCHAR , ArrDelay NUMBER(38, 2) , ArrDelayMinutes NUMBER(38, 2) , ArrDel15 NUMBER(38, 2) , ArrivalDelayGroups NUMBER(38, 0) , ArrTimeBlk VARCHAR , Cancelled NUMBER(38, 2) , CancellationCode VARCHAR , Diverted NUMBER(38, 2) , CRSElapsedTime NUMBER(38, 2) , ActualElapsedTime NUMBER(38, 2) , AirTime NUMBER(38, 2) , Flights NUMBER(38, 2) , Distance NUMBER(38, 2) , DistanceGroup NUMBER(38, 0) , CarrierDelay NUMBER(38, 2) , WeatherDelay NUMBER(38, 2) , NASDelay NUMBER(38, 2) , SecurityDelay NUMBER(38, 2) , LateAircraftDelay NUMBER(38, 2) , FirstDepTime VARCHAR , TotalAddGTime NUMBER(38, 2) , LongestAddGTime NUMBER(38, 2) , DivAirportLandings NUMBER(38, 0) , DivReachedDest NUMBER(38, 2) , DivActualElapsedTime NUMBER(38, 2) , DivArrDelay NUMBER(38, 2) , DivDistance NUMBER(38, 2) , Div1Airport VARCHAR , Div1AirportID NUMBER(38, 0) , Div1AirportSeqID NUMBER(38, 0) , Div1WheelsOn VARCHAR , Div1TotalGTime NUMBER(38, 2) , Div1LongestGTime NUMBER(38, 2) , Div1WheelsOff VARCHAR , Div1TailNum VARCHAR , Div2Airport VARCHAR , Div2AirportID VARCHAR , Div2AirportSeqID VARCHAR , Div2WheelsOn VARCHAR , Div2TotalGTime VARCHAR , Div2LongestGTime VARCHAR , Div2WheelsOff VARCHAR , Div2TailNum VARCHAR , Div3Airport VARCHAR , Div3AirportID VARCHAR , Div3AirportSeqID VARCHAR , Div3WheelsOn VARCHAR , Div3TotalGTime VARCHAR , Div3LongestGTime VARCHAR , Div3WheelsOff VARCHAR , Div3TailNum VARCHAR , Div4Airport VARCHAR , Div4AirportID VARCHAR , Div4AirportSeqID VARCHAR , Div4WheelsOn VARCHAR , Div4TotalGTime VARCHAR , Div4LongestGTime VARCHAR , Div4WheelsOff VARCHAR , Div4TailNum VARCHAR , Div5Airport VARCHAR , Div5AirportID VARCHAR , Div5AirportSeqID VARCHAR , Div5WheelsOn VARCHAR , Div5TotalGTime VARCHAR , Div5LongestGTime VARCHAR , Div5WheelsOff VARCHAR , Div5TailNum VARCHAR ); 

CREATE TEMP FILE FORMAT "AVIATION_WEATHER_PROJECT"."PUBLIC"."temp_file_format_2026-10-15T19:47:40.592Z"
	TYPE=CSV
    SKIP_HEADER=1
    FIELD_DELIMITER=','
    TRIM_SPACE=TRUE
    FIELD_OPTIONALLY_ENCLOSED_BY='"'
    REPLACE_INVALID_CHARACTERS=TRUE
    DATE_FORMAT=AUTO
    TIME_FORMAT=AUTO
    TIMESTAMP_FORMAT=AUTO; 

COPY INTO "AVIATION_WEATHER_PROJECT"."PUBLIC"."FLIGHTS" 
FROM (SELECT $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12, $13, $14, $15, $16, $17, $18, $19, $20, $21, $22, $23, $24, $25, $26, $27, $28, $29, $30, $31, $32, $33, $34, $35, $36, $37, $38, $39, $40, $41, $42, $43, $44, $45, $46, $47, $48, $49, $50, $51, $52, $53, $54, $55, $56, $57, $58, $59, $60, $61, $62, $63, $64, $65, $66, $67, $68, $69, $70, $71, $72, $73, $74, $75, $76, $77, $78, $79, $80, $81, $82, $83, $84, $85, $86, $87, $88, $89, $90, $91, $92, $93, $94, $95, $96, $97, $98, $99, $100, $101, $102, $103, $104, $105, $106, $107, $108, $109
	FROM '@"AVIATION_WEATHER_PROJECT"."PUBLIC"."STAGING_DATA"') 
FILES = ('On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1.csv.gz') 
FILE_FORMAT = '"AVIATION_WEATHER_PROJECT"."PUBLIC"."temp_file_format_2026-10-15T19:47:40.592Z"' 
ON_ERROR=ABORT_STATEMENT 
-- For more details, see: https://docs.snowflake.com/en/sql-reference/sql/copy-into-table